# Gait-ViViT: A Video Processing Model for Parkinson's Disease Detection

In [ ]:
# Required libraries.
import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.v2 as v2
from torchvision import tv_tensors
import torch.nn as nn
import torch.nn.functional as F
import math
from transformers import VivitModel, VivitForVideoClassification
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix
from torch.amp import GradScaler, autocast
import matplotlib.pyplot as plt
import seaborn as sns
import torch.optim as optim
from torch.optim import lr_scheduler
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from transformers import get_cosine_schedule_with_warmup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Remember to first define the `GaitViViTDataset` class and the `GaitCNNLSTM` model.

In [ ]:
class GaitViViTDataset(Dataset):
  def __init__(self, tensor_df, frames_per_video=32, transform=None):
    # Load the dataframe for initialization.
    if isinstance(tensor_df, str):
      self.data = pd.read_csv(tensor_df)
    else:
      self.data = tensor_df.reset_index(drop=True)
    self.frames_per_video = frames_per_video
    self.transform = transform

  def __len__(self):
    # Find the number of elements in the dataframe.
    return len(self.data)

  def __getitem__(self, idx):
    # Load, and eventually transform, a specific tensor and recover the label associated to the original video.
    tensor_path = self.data.iloc[idx]["path"]
    tensor = torch.load(tensor_path)

    # Convert the tensor from uint8 to float32 and permute its shape from (F, H, W, C) to (C, F, H, W) for training compatibility.
    tensor = tensor.float()
    tensor = tensor.permute(3, 0, 1, 2).contiguous()
    tensor_shape = tensor.shape

    # Apply ImageNet normalization.
    if tensor.max() > 1:
      tensor = tensor / 255.0 # Scaling.
    mean = tensor.new_tensor([0.485, 0.456, 0.406]).view(3, 1, 1, 1) # Make mean broadcast-compatible.
    std = tensor.new_tensor([0.229, 0.224, 0.225]).view(3, 1, 1, 1) # Make std broadcast-compatible.
    tensor = (tensor - mean) / std # Normalization.

    if self.transform:
      # Apply any transformation to the loaded tensor.
      tensor = tv_tensors.Video(tensor) # Make sure the transformation is applied to ALL frames.
      tensor = self.transform(tensor)
    label = self.data.iloc[idx]["parkinson"]
    return tensor, label

In [ ]:
class AnomalyHead(nn.Module):
  def __init__(self, in_features, hidden_features):
    super().__init__()

    self.fc1 = nn.Linear(in_features, hidden_features)
    self.act = nn.ReLU()
    # self.act = nn.GELU() # For Version 4.
    self.fc2 = nn.Linear(hidden_features, 1)

  def forward(self, x):
    x = self.fc1(x)
    x = self.act(x)
    logit = self.fc2(x)
    return logit

In [ ]:
class GaitCNNLSTM(nn.Module):
  def __init__(self, hidden_features=256, num_layers=2, dropout=0.1, cls_ratio=0.5):
    super().__init__()

    # Define the CNN + LSTM backbone.
    resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    self.cnn = nn.Sequential(*list(resnet.children())[:-1])
    self.lstm = nn.LSTM(input_size=512, hidden_size=hidden_features, num_layers=num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0.0)

    # Attach the custom classification head.
    self.hidden_dim = int(hidden_features * cls_ratio)
    self.head = AnomalyHead(in_features=hidden_features, hidden_features=self.hidden_dim)

  def forward(self, x):
    # Reshape data to shape (B * T, C, H, W).
    B, C, T, H, W = x.shape
    x = x.permute(0, 2, 1, 3, 4)
    x = x.reshape(B * T, C, H, W)

    # Extract spatial features by passing through the CNN.
    cnn_out = self.cnn(x)
    cnn_out = cnn_out.view(B, T, -1) # Reshape to (B, T, 512).

    # Extract temporal features by passing through the LSTM.
    lstm_out, (hn, cn) = self.lstm(cnn_out)

    # Extract the last hidden state that will be passed to the classification head.
    last_out = lstm_out[:, -1, :]
    logit = self.head(last_out)
    return logit

## 4 - Training

Since the goal of this project is to adapt the *Video Vision Transformer* to a supervised anomaly detection task, the model will be trained using a hybrid **transfer learning** approach that combines linear probing and model fine-tuning.

### Data Split

The dataset will be split into a **training dataset**, a **validation dataset** and a **testing dataset**.

| Subset | Size |
| :---: | :---:|
| Training | $70\%$ |
| Validation | $10\%$ |
| Testing | $20\%$ |

To avoid data leakage, the training, validation and testing datasets are constructed using the `StratifiedKFold` function, which combines **$K$-fold cross-validation** and **stratified sampling** to ensure representative splits across all folds.

In [ ]:
def split_data(df):
  # In order to avoid data leakage, extract patient IDs and the respective diagnosis.
  patients = df[["patient_id", "parkinson"]].drop_duplicates().reset_index(drop=True)
  X_p = patients["patient_id"].values
  y_p = patients["parkinson"].values
  fold_indices = {}

  # Outer split: 20% is isolated as the testing set, the remaining 80% will be used to build the training and validation set.
  outer_split = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

  for i, (train_val_p_idx, test_p_idx) in enumerate(outer_split.split(X_p, y_p)):
    # Determine which patients belong to the testing set and which patients will belong to the training or validation set.
    train_val_patients = X_p[train_val_p_idx]
    test_patients = X_p[test_p_idx]
    y_train_val_p = y_p[train_val_p_idx]

    # Inner split: Around 10% of the original dataset is isolated as the validation set, the rest will compose the training set.
    inner_split = StratifiedKFold(n_splits=7, shuffle=True, random_state=42)
    train_p_idx, val_p_idx = next(inner_split.split(train_val_patients, y_train_val_p))
    train_patients = train_val_patients[train_p_idx]
    val_patients = train_val_patients[val_p_idx]

    # Determine the indices of the training, validation and testing sets.
    train_idx = df.index[df["patient_id"].isin(train_patients)].tolist()
    val_idx = df.index[df["patient_id"].isin(val_patients)].tolist()
    test_idx = df.index[df["patient_id"].isin(test_patients)].tolist()

    # Save the indices for the current fold.
    fold_indices[i] = {"train": train_idx, "val": val_idx, "test": test_idx}

  return fold_indices

In [ ]:
# Load the dataframe and determine the splits.
data = pd.read_csv("/content/drive/MyDrive/bachelor_thesis/dataframes/tensor_dataset.csv")
folds = split_data(data)

### Training Loop

Each epoch of the training loop consists of a **training step**, where the model performs a forward pass and uses the results to adjust its weights during the backward pass, and a **validation step**, where the model performs an unbiased forward pass.

**N.B.:** To avoid computational overhead, training is performed using gradient accumulation and mixed precision arithmetic.

In [ ]:
def get_metrics(y_true, y_probs, threshold=0.5):
  y_preds = (y_probs >= threshold).astype(int) # Convert from boolean to integer.

  # Compute each metric.
  accuracy = accuracy_score(y_true, y_preds)
  precision = precision_score(y_true, y_preds, zero_division=0)
  recall = recall_score(y_true, y_preds, zero_division=0)
  f1 = f1_score(y_true, y_preds, zero_division=0)
  try:
    roc_auc = roc_auc_score(y_true, y_probs)
  except ValueError:
    roc_auc = 0.0

  return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1, "roc_auc": roc_auc}

In [ ]:
def training_loop(model, dataloader, loss_function, optimizer, device, accumulation_steps=2):
  model.train() # Set the model in training mode.
  running_loss = 0.0
  all_targets, all_probs = [], [] # Needed to compute metrics.

  # Reset gradients.
  optimizer.zero_grad()

  for batch_idx, (inputs, targets) in enumerate(dataloader):
    # Move data to the chosen device.
    inputs, targets = inputs.to(device), targets.float().to(device) # Labels are converted to float for compatibility with BCEWithLogitsLoss.

    # Run a forward pass using mixed precision.
    with autocast("cuda"):
      logits = model(inputs).squeeze(-1)
      loss = loss_function(logits, targets)
      loss = loss / accumulation_steps
    scaler.scale(loss).backward()

    # Run the backward pass whenever needed.
    if (batch_idx + 1) % accumulation_steps == 0 or (batch_idx + 1) == len(dataloader):
      scaler.step(optimizer)
      scaler.update()
      optimizer.zero_grad()

    # Update the running loss and save data to get metrics.
    running_loss += (loss.item() * accumulation_steps) * inputs.size(0)
    probs = torch.sigmoid(logits).detach().cpu().numpy()
    all_probs.extend(probs)
    all_targets.extend(targets.detach().cpu().numpy())

  epoch_loss = running_loss / len(dataloader.dataset) # Average across batch losses.
  metrics = get_metrics(np.array(all_targets), np.array(all_probs))
  metrics["loss"] = epoch_loss

  return metrics

In [ ]:
def validation_loop(model, dataloader, loss_function, device, threshold=0.5, return_raw=False):
  model.eval() # Set the model in evaluation mode, disabling stochastic patterns.
  running_loss = 0.0
  all_targets, all_probs = [], [] # Needed to compute metrics.

  with torch.no_grad():
    for inputs, targets in dataloader:
      # Move data to the chosen device.
      inputs, targets = inputs.to(device), targets.float().to(device) # Labels are converted to float for compatibility with BCEWithLogitsLoss.

      # Run a forward pass using mixed precision.
      with autocast("cuda"):
        logits = model(inputs).squeeze(-1)
        loss = loss_function(logits, targets)

      # Update the running loss and save data to get metrics.
      running_loss += loss.item() * inputs.size(0)
      probs = torch.sigmoid(logits).detach().cpu().numpy()
      all_probs.extend(probs)
      all_targets.extend(targets.detach().cpu().numpy())

    epoch_loss = running_loss / len(dataloader.dataset) # Average across batch losses.
    metrics = get_metrics(np.array(all_targets), np.array(all_probs), threshold=threshold)
    metrics["loss"] = epoch_loss

  if return_raw:
    return metrics, np.array(all_targets), np.array(all_probs)
  else:
    return metrics

### Linear Probing

The first step of the training procedure is the **linear probing** phase, which consists of freezing the backbone and training just the final classification head.

This step lasts **5 epochs** and the model is trained using **stochastic gradient descent with momentum** and **cosine annealing scheduling with linear warmups**.

In [ ]:
def linear_probing(model, dataloaders, loss_function, num_epochs=5, model_path="/content/drive/MyDrive/bachelor_thesis/models/best_head.pth", metrics_path="/content/drive/MyDrive/bachelor_thesis/metrics/head_lp.csv", device="cpu", checkpoint_path="/content/drive/MyDrive/bachelor_thesis/models/lp_resume_checkpoint.pth"):
  lp_history = [] # To save the metrics.
  best_f1 = 0.0 # Due to class imbalance, F1-score tends to be a more reliable metric.

  # Freeze the backbone to train just the head.
  for name, param in model.named_parameters():
    if "head" not in name:
      param.requires_grad = False
    else:
      param.requires_grad = True

  # Define the optimizer and the scheduler.
  optimizer_lp = torch.optim.SGD(model.head.parameters(), lr=3e-04, momentum=0.9) # Use a higher learning rate for this stage.
  # scheduler_lp = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_lp, T_max=num_epochs, eta_min=1e-5)
  training_steps = (len(dataloaders["train"]) * num_epochs) // 2 # 2 is the number of gradient accumulation steps.
  scheduler_lp = get_cosine_schedule_with_warmup(optimizer_lp, num_warmup_steps = int(0.1 * training_steps), num_training_steps=training_steps) # Apply warmup for 10% of the training.

  # Recover the latest checkpoint.
  start_epoch = 0
  if os.path.exists(checkpoint_path):
    print("Resuming Linear Probing.")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer_lp.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler_lp.load_state_dict(checkpoint["scheduler_state_dict"])
    best_f1 = checkpoint["best_f1"]
    lp_history = checkpoint["lp_history"]
    start_epoch = checkpoint["epoch"]

  # Run training and validation through each epoch.
  for epoch in range(start_epoch, num_epochs):
    # Get the training and validation metrics.
    train_metrics = training_loop(model, dataloaders["train"], loss_function, optimizer_lp, device)
    val_metrics = validation_loop(model, dataloaders["val"], loss_function, device)

    # Update the learning rate for next epoch.
    scheduler_lp.step()

    # Save the weights whenever the validation F1-score reaches a new global maximum.
    if val_metrics["f1"] > best_f1:
      best_f1 = val_metrics["f1"]
      torch.save(model.state_dict(), model_path)

    # Save the metrics for the current epoch.
    row = {"epoch": epoch + 1, "phase": "lp", "lr": scheduler_lp.get_last_lr()[0]}
    row.update({f"train_{k}": v for k, v in train_metrics.items()})
    row.update({f"val_{k}": v for k, v in val_metrics.items()})
    lp_history.append(row)

    # Save the checkpoint.
    torch.save({"epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer_lp.state_dict(),
                "scheduler_state_dict": scheduler_lp.state_dict(),
                "best_f1": best_f1,
                "lp_history": lp_history}, checkpoint_path)

  # Save the complete metrics.
  df_lp = pd.DataFrame(lp_history)
  df_lp.to_csv(metrics_path, index=False)

  return model

### Full Finetuning

After the linear probing phase, the **fine-tuning** phase unfreezes the entire model to adapt it to the new task.

This step lasts **15 epochs** and the model is trained using **stochastic gradient descent with momentum and weight decay** and **cosine annealing scheduling**.

Since fine-tuning tends to be computationally expensive and can lead to overfitting, the model is trained using **grid search**, testing different combinations of starting learning rate and weight decay, and introducing an **early stopping counter** before the model starts overfitting.

In [ ]:
def model_finetuning(model, dataloaders, loss_function, num_epochs=15, head_path="/content/drive/MyDrive/bachelor_thesis/models/best_head.pth", model_path="/content/drive/MyDrive/bachelor_thesis/models/best_model.pth", metrics_path="/content/drive/MyDrive/bachelor_thesis/metrics/model_ft.csv", device="cpu", checkpoint_path="/content/drive/MyDrive/bachelor_thesis/models/ft_resume_checkpoint.pth"):
  ft_history = [] # To save the metrics.
  global_best_f1 = 0.0 # Due to class imbalance, F1-score tends to be a more reliable metric.

  # Define the parameter grid, focusing on the starting learning rate and the weight decay.
  parameters = {
      "start_lr": [1e-6, 1e-5, 1e-4],
      "weight_decay": [1e-2, 0.1]
  }
  grid = ParameterGrid(parameters)

  # Recover the latest checkpoint.
  start_run_idx = 0 # To keep track of the hyperparameter combinations.
  start_epoch = 0 # To keep track of the current epoch.
  if os.path.exists(checkpoint_path):
    print("Resuming Finetuning.")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    start_run_idx = checkpoint["run_idx"]
    start_epoch = checkpoint["epoch"]
    ft_history = checkpoint["ft_history"]
    global_best_f1 = checkpoint["global_best_f1"]

  # Iterate through the parameter configurations.
  for idx, params in enumerate(grid):
    if idx < start_run_idx:
      # Skip completed runs.
      continue

    print(f"Run {idx + 1} | Parameters: {params}")

    # Define the parameters for early stopping.
    patience = 5
    tolerance = 0.001

    # Determine whether to resume the current run or reset everything for a new run.
    if idx == start_run_idx and os.path.exists(checkpoint_path):
      # Resume the current run: Load the latest model and the most recent metrics for the run.
      print(f"Resuming Run {idx + 1}.")
      checkpoint = torch.load(checkpoint_path, map_location=device)
      model.load_state_dict(checkpoint["model_state_dict"])
      run_best_f1 = checkpoint["run_best_f1"]
      counter = checkpoint["counter"]
      current_start_epoch = start_epoch
    elif idx > start_run_idx or not os.path.exists(checkpoint_path):
      # Reset for a new run: Load the weights of the head and unfreeze the backbone, resetting the run.
      model.load_state_dict(torch.load(head_path, map_location=device))
      for param in model.parameters():
        param.requires_grad = True
      run_best_f1 = 0.0
      counter = 0
      current_start_epoch = 0

    # Define the optimizer and the scheduler.
    optimizer_ft = torch.optim.SGD(model.parameters(), lr=params["start_lr"], momentum=0.9, weight_decay=params["weight_decay"])
    scheduler_ft = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_ft, T_max=num_epochs, eta_min=1e-7)
    if idx == start_run_idx and os.path.exists(checkpoint_path):
      # Resume the current run: Load the latest optimizer and scheduler states.
      optimizer_ft.load_state_dict(checkpoint["optimizer_state_dict"])
      scheduler_ft.load_state_dict(checkpoint["scheduler_state_dict"])

    # Run training and validation through each epoch.
    for epoch in range(current_start_epoch, num_epochs):
      # Get the training and validation metrics.
      train_metrics = training_loop(model, dataloaders["train"], loss_function, optimizer_ft, device)
      val_metrics = validation_loop(model, dataloaders["val"], loss_function, device)

      # Update the learning rate for the next epoch.
      scheduler_ft.step()

      # Save the metrics for the current epoch.
      row = {"epoch": epoch + 1, "phase": "ft", "lr": scheduler_ft.get_last_lr()[0]}
      row.update(params)
      row.update({f"train_{k}": v for k, v in train_metrics.items()})
      row.update({f"val_{k}": v for k, v in val_metrics.items()})
      ft_history.append(row)

      current_f1 = val_metrics["f1"]

      # Save the weights whenever the validation F1-score reaches a new global maximum.
      if current_f1 > global_best_f1:
        global_best_f1 = current_f1
        torch.save(model.state_dict(), model_path)

      # Check for early stopping.
      if current_f1 > run_best_f1 + tolerance:
        # Significant improvement: Reset the counter.
        run_best_f1 = current_f1
        counter = 0
      else:
        # No significant improvement: Update the counter.
        counter += 1

      # Save the checkpoint.
      torch.save({"run_idx": idx,
                  "epoch": epoch + 1,
                  "model_state_dict": model.state_dict(),
                  "optimizer_state_dict": optimizer_ft.state_dict(),
                  "scheduler_state_dict": scheduler_ft.state_dict(),
                  "global_best_f1": global_best_f1,
                  "run_best_f1": run_best_f1,
                  "ft_history": ft_history,
                  "counter": counter}, checkpoint_path)

      # Check for early stopping.
      if counter >= patience:
        print(f"Early stopping triggered at epoch {epoch + 1} for parameters {params}.")
        break

  # Save the complete metrics.
  df_ft = pd.DataFrame(ft_history)
  df_ft.to_csv(metrics_path, index=False)

  return model

### Testing Stage

After the linear probing and fine-tuning steps, the **best-performing model** performs the testing stage in order to assess its learning ability on unseen data.

In [ ]:
def save_confusion_matrix(y_true, y_probs, threshold=0.5, save_path="/content/drive/MyDrive/bachelor_thesis/metrics/confusion_matrix.png"):
  # Convert logits into predictions.
  y_preds = (y_probs >= threshold).astype(int)

  # Compute and plot the confusion matrix.
  cm = confusion_matrix(y_true, y_preds)
  plt.figure(figsize=(8, 6))
  sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
              xticklabels=['Healthy', 'Parkinson'],
              yticklabels=['Healthy', 'Parkinson'])

  plt.title(f'Confusion Matrix')
  plt.xlabel('Predicted Label')
  plt.ylabel('True Label')

  # Save the image.
  plt.savefig(save_path, dpi=300, bbox_inches='tight')
  plt.close()

In [ ]:
def model_test(model, dataloaders, loss_function, best_model_path="/content/drive/MyDrive/bachelor_thesis/models/best_model.pth", metrics_path="/content/drive/MyDrive/bachelor_thesis/metrics/model_test.csv", matrix_path="/content/drive/MyDrive/bachelor_thesis/metrics/model_test_cm.png", device="cpu"):
  # Load the best model and run a test loop.
  model.load_state_dict(torch.load(best_model_path, map_location=device))
  test_metrics, y_true, y_probs = validation_loop(model, dataloaders["test"], loss_function, device, threshold=0.5, return_raw=True)

  # Save the metrics.
  data = []
  data.append(test_metrics)
  df = pd.DataFrame(data)
  df.to_csv(metrics_path, index=False)

  # Get the confusion matrix.
  save_confusion_matrix(y_true, y_probs, threshold=0.5, save_path=matrix_path)

  # Print the results.
  for k, v in test_metrics.items():
    print(f"Test {k}: {v}.")

  return test_metrics, y_true, y_probs

### Ablation Study

Due to the medical purpose of the model, an ablation study is carried out in order to understand which classification threshold would be more suitable for this task.

| Threshold | Advantages | Disadvantages |
| :---: | :---: | :---: |
| Below $0.5$ | The model is more likely to detect early/mild symptoms | The model is vulnerable to false positives |
| Above $0.5$ | The model is more robust against false positives | The model might overlook early/mild symptoms |

In [ ]:
def ablation_study(y_true, y_probs, thresholds=np.arange(0.1, 1.0, 0.1), csv_path="/content/drive/MyDrive/bachelor_thesis/metrics/threshold_ablation.csv", plot_path="/content/drive/MyDrive/bachelor_thesis/metrics/threshold_plot.png"):
  ablation_results = []

  # Run the ablation study.
  for t in thresholds:
    metrics = get_metrics(y_true, y_probs, threshold=t)
    metrics["threshold"] = t
    ablation_results.append(metrics)

  ablation_df = pd.DataFrame(ablation_results)
  ablation_df.to_csv(csv_path, index=False)

  # Create a plot for the ablation study.
  plt.figure(figsize=(10, 6))
  sns.set_style("whitegrid")

  # Draw the metrics.
  plt.plot(ablation_df["threshold"], ablation_df["f1"], marker='o', label="F1-Score", color="green")
  plt.plot(ablation_df["threshold"], ablation_df["precision"], marker='s', label="Precision", color="blue")
  plt.plot(ablation_df["threshold"], ablation_df["recall"], marker='^', label="Recall", color="orange")
  plt.axhline(y=ablation_df["roc_auc"].iloc[0], color='r', linestyle=':', label=f'ROC-AUC ({ablation_df["roc_auc"].iloc[0]:.3f})')

  # Add the final details.
  plt.title("Ablation Study: Performance vs Threshold")
  plt.xlabel("Classification Threshold")
  plt.ylabel("Score")
  plt.legend(loc="lower center")
  plt.savefig(plot_path, dpi=300)

  return ablation_df

### Main Loop

Overall, the model is trained using $K$-fold cross-validation, which comes in handy to obtain more reliable results, and taking binary cross-entropy loss in order to address the dataset's class imbalance.

For each generated fold, the **initialized model undergoes linear probing and fine-tuning**, saving the best performing model, and, after the first two steps, the best-performing model is used to perform the **testing stage** and the **ablation study**.

**N.B.:** The following loop uses a generic default configuration for saving models and metrics.

In [ ]:
# Choose which device to use for the training procedure.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}.")

# Define the loss function.
weight = 947 / 115 # Weights taken from the class distribution in tensor_dataset.csv.
pos_weight = torch.tensor([weight]).to(device)
loss_function = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Create, if needed, the directories to store model weights and metrics.
os.makedirs("/content/drive/MyDrive/bachelor_thesis/models", exist_ok=True)
os.makedirs("/content/drive/MyDrive/bachelor_thesis/metrics", exist_ok=True)

In [ ]:
# Iterate through folds.
for k, v in folds.items():
  # Since the procedure may interrupt, skip any completed folds.
  final_ablation_path = f"/content/drive/MyDrive/bachelor_thesis/metrics/ablation_study_fold_{k + 1}.csv"
  if os.path.exists(final_ablation_path):
    print(f"Fold {k + 1} previously completed. Skipping to the next one.")
    continue

  # Determine the splits.
  train_idx, val_idx, test_idx = v["train"], v["val"], v["test"]
  train_data = data.iloc[train_idx].copy()
  val_data = data.iloc[val_idx].copy()
  test_data = data.iloc[test_idx].copy()

  # Create the datasets and the corresponding dataloaders.
  train_dataset = GaitViViTDataset(tensor_df=train_data, frames_per_video=32)
  val_dataset = GaitViViTDataset(tensor_df=val_data, frames_per_video=32)
  test_dataset = GaitViViTDataset(tensor_df=test_data, frames_per_video=32)

  train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
  val_dataloader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)
  test_dataloader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)
  dataloaders = {"train": train_dataloader, "val": val_dataloader, "test": test_dataloader}

  # Create an instance of the GaitViViT model.
  model = GaitCNNLSTM().to(device)

  # Perform linear probing.
  print("--- Phase 1 - Linear Probing ---")
  model = linear_probing(model,
                         dataloaders,
                         loss_function,
                         num_epochs=5,
                         model_path=f"/content/drive/MyDrive/bachelor_thesis/models/best_head_fold_{k + 1}.pth",
                         metrics_path=f"/content/drive/MyDrive/bachelor_thesis/metrics/head_lp_fold_{k + 1}.csv",
                         device=device,
                         checkpoint_path=f"/content/drive/MyDrive/bachelor_thesis/models/lp_resume_checkpoint_fold_{k + 1}.pth")

  # Perform fine-tuning.
  print("--- Phase 2 - Fine-tuning ---")
  model = model_finetuning(model,
                           dataloaders,
                           loss_function,
                           num_epochs=15,
                           head_path=f"/content/drive/MyDrive/bachelor_thesis/models/best_head_fold_{k + 1}.pth",
                           model_path=f"/content/drive/MyDrive/bachelor_thesis/models/best_model_fold_{k + 1}.pth",
                           metrics_path=f"/content/drive/MyDrive/bachelor_thesis/metrics/model_ft_fold_{k + 1}.csv",
                           device=device,
                           checkpoint_path=f"/content/drive/MyDrive/bachelor_thesis/models/ft_resume_checkpoint_fold_{k + 1}.pth")

  # Perform testing.
  print("--- Phase 3 - Testing ---")
  test_results, y_true, y_probs = model_test(model,
                                             dataloaders,
                                             loss_function,
                                             best_model_path=f"/content/drive/MyDrive/bachelor_thesis/models/best_model_fold_{k + 1}.pth",
                                             metrics_path=f"/content/drive/MyDrive/bachelor_thesis/metrics/model_test_fold_{k + 1}.csv",
                                             matrix_path=f"/content/drive/MyDrive/bachelor_thesis/metrics/confusion_matrix_fold_{k + 1}.png",
                                             device=device)

  # Perform the ablation study.
  print("--- Phase 4 - Ablation Study ---")
  ablation_df = ablation_study(y_true,
                               y_probs,
                               thresholds=np.arange(0.3, 0.8, 0.1),
                               csv_path=f"/content/drive/MyDrive/bachelor_thesis/metrics/ablation_study_fold_{k + 1}.csv",
                               plot_path=f"/content/drive/MyDrive/bachelor_thesis/metrics/ablation_study_fold_{k + 1}.png")

  print(f"--- Fold {k + 1} successfully completed. ---")
  torch.cuda.empty_cache()

print("Training successfully completed.")